# Week 1: End-to-End Data Science Project Workflow

## Learning Objectives
- Structure a complete data science project
- Apply the full ML pipeline from data to predictions
- Create reproducible experiments
- Document findings professionally

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
print('All libraries imported')

## 1. Project Structure

A well-structured data science project follows this layout:

```
project/
   data/
      raw/          # Original, immutable data
      processed/    # Cleaned, transformed data
   notebooks/      # Exploratory analysis
   src/
      data.py       # Data loading and processing
      features.py   # Feature engineering
      models.py     # Model training
      evaluate.py   # Evaluation utilities
   models/         # Saved model artifacts
   reports/        # Generated analysis
   requirements.txt
   README.md
```

## 2. Step 1 — Data Loading and Exploration

In [ ]:
housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = pd.Series(housing.target, name='MedHouseVal')

print('Dataset shape:', X.shape)
print('\nFeature descriptions:')
for feat, desc in zip(housing.feature_names, [
    'Median income', 'House age', 'Avg rooms', 'Avg bedrooms',
    'Population', 'Avg occupancy', 'Latitude', 'Longitude']):
    print(f'  {feat}: {desc}')

print('\nTarget stats:')
print(y.describe().round(3))

## 3. Step 2 — EDA

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, col in enumerate(X.columns):
    axes[i//4, i%4].hist(X[col], bins=30, edgecolor='white')
    axes[i//4, i%4].set_title(col)
plt.suptitle('Feature Distributions')
plt.tight_layout(); plt.show()

plt.figure(figsize=(10, 7))
corr = pd.concat([X, y], axis=1).corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Matrix')
plt.tight_layout(); plt.show()

## 4. Step 3 — Model Training and Evaluation

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

models = {
    'Ridge Regression': Pipeline([('s', StandardScaler()), ('m', Ridge())]),
    'Random Forest':    RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boost':   GradientBoostingRegressor(n_estimators=100, random_state=42)
}

results = {}
for name, model in models.items():
    cv  = cross_val_score(model, X_train, y_train, cv=5, scoring='r2').mean()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    results[name] = {'CV R2': cv, 'Test RMSE': rmse, 'Test R2': r2}
    print(f'{name:20s}: CV R2={cv:.4f}, RMSE={rmse:.4f}, R2={r2:.4f}')

best_name = max(results, key=lambda k: results[k]['Test R2'])
best_model = models[best_name]
print(f'\nBest model: {best_name}')

## 5. Step 4 — Save and Load Model

In [ ]:
import pickle

# Save model
with open('/tmp/best_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

# Load and verify
with open('/tmp/best_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

y_loaded_pred = loaded_model.predict(X_test)
assert np.allclose(y_pred, y_loaded_pred), 'Loaded model gives different predictions!'
print('Model saved and loaded successfully')

## Exercise

1. Add feature engineering steps (interaction terms, log transforms)
2. Tune the best model's hyperparameters with RandomizedSearchCV
3. Write a brief project report summarising findings in markdown

In [ ]:
# Your code here


## Summary

- ✓ Project folder structure
- ✓ Complete EDA to model pipeline
- ✓ Model comparison and selection
- ✓ Saving and loading models with pickle

## Next Week
Model Deployment Strategies!